# `sbi`
We'll explore SBI techniques such as neural posterior estimation and neural likelihood estimation using the `sbi` python package: https://sbi.readthedocs.io/en/latest/. We'll go through the sample inference examples as the SBI with variational inference demo.

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import corner

To install the `sbi` package use
```
python -m pip install sbi
```
in your conda environment. Beware of some `torch` and `jupyter` version requirements that may conflict.

In [ ]:
import sbi
import torch

In [ ]:
np.random.seed(10)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Example 0
Let's return to the example from a few weeks ago

> We point our telescope to observe the light from a single star with true flux $F_{\rm true}$, which remains constant with time. We will ignore any systematic errors. We perform $N$ measurements with our telescope, where the $i^{\rm th}$ measurement report an observed flux $F_i$ with uncertainty/error $e_i$.

Lets generate a synthetic observation

In [ ]:
F_true = 1000
F_obs = np.random.poisson(lam=F_true)

In [ ]:
# simulator of the data generation process
def simulation(F_true): 
    return np.random.poisson(lam=F_true)

For our prior, lets use a simple Gaussian
$$p(\theta) = \mathcal{N}(1002, 4)$$
This would hypothetically be measurement from a prior experiment

In [ ]:
def sample_prior(): 
    # sample prior
    return 1002. + 4. * np.random.normal()

Let's sample $p(\theta, \mathcal{D})$

In [ ]:
Nsample = 10000

samples = np.zeros((Nsample, 2))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.array([theta_p, data_p])
samples = torch.tensor(samples, dtype=torch.float32)

In [ ]:
# plot p(theta, D)
fig = plt.figure(figsize=(3,3))
sub = fig.add_subplot(111)
sub.scatter(samples[:,0].numpy(), samples[:,1].numpy(), s=1, c='k')
sub.set_xlabel(r'$\theta$', fontsize=25)
sub.set_ylabel('$\mathcal{D}$', fontsize=25); plt.show()

Now lets use 
## neural posterior estimation
to directly estimate the posterior with a neural density estimator

In [ ]:
from sbi.inference import NPE

`sbi` package has a convenient `NPE` inference object. You can specify the neural density estimator. Here we'll initially use MADE

In [ ]:
made = NPE(density_estimator='made')

Now we can provide the `NPE` the simulations, or samples $\{(\theta', \mathcal{D}')\}$

In [ ]:
_ = made.append_simulations(samples[:,0][:,None], samples[:,1][:,None])

Then, train the NPE

In [ ]:
npe_made = made.train()

After training the NPE, we can performance inference as follows

In [ ]:
posterior_made = made.build_posterior()
print(posterior_made)

In [ ]:
q_samples0 = posterior_made.sample((10000,), x=F_obs)

In [ ]:
plt.figure(figsize=(4,2))
plt.hist(q_samples0, range=(960, 1040), bins=40)
plt.axvline(F_true, color='k', linestyle='--'); plt.show()

Since the `sbi` package makes it very convenient to try different neural density estimators, now lets try with MAF. 

In [ ]:
maf = NPE(density_estimator='maf')

# append (theta, D) from the simulations
_ = maf.append_simulations(torch.tensor(samples[:,0][:,None], dtype=torch.float32), 
                           torch.tensor(samples[:,1][:,None], dtype=torch.float32))
# train NPE. 
npe_maf = maf.train()

Then the inference

In [ ]:
posterior_maf = maf.build_posterior()
q_samples1 = posterior_made.sample((10000,), x=F_obs)

plt.figure(figsize=(4,2))
plt.hist(q_samples0, range=(960, 1040), bins=40)
plt.hist(q_samples1, range=(960, 1040), bins=40, alpha=0.8)
plt.axvline(F_true, color='k', linestyle='--'); plt.show()

# Example 1
Now lets fits a line to some data with Gaussian priors (same as the SBI + VI demo)

In [ ]:
a_true, b_true = 0, 1.75
x = np.array([1, 1.5, 2, 2.5])
y = np.array([1.87640523, 2.59001572, 3.4978738 , 4.47408932])
yerr = 0.1

In [ ]:
plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k')

In [ ]:
def simulation(theta): 
    a, b = theta
    noise = yerr * np.random.normal(size=len(x))
    return a + b * x + noise

def sample_prior(): 
    # sample a and b from uniform prior
    _a = -0.5 + np.random.normal() 
    _b = 5 + 3*np.random.normal()
    return [_a, _b]

same as above, sample $(\theta', \mathcal{D}') \sim p(\theta, \mathcal{D})$

In [ ]:
Nsample = 500000

samples = np.zeros((Nsample, 6))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.concatenate([theta_p, data_p])
samples = torch.tensor(samples, dtype=torch.float32)

Now train the NPE

In [ ]:
maf = NPE(density_estimator='maf')

# append (theta, D) from the simulations
_ = maf.append_simulations(samples[:,:2], samples[:,2:])
# train NPE. 
npe_maf = maf.train()

In [ ]:
posterior_maf = maf.build_posterior()
q_samples = posterior_maf.sample((10000,), x=torch.tensor(y, dtype=torch.float32))

In [ ]:
fig = plt.figure(figsize=(4,4))
fig = corner.corner(q_samples.numpy(), labels=['a', 'b'], fig=fig)
corner.overplot_lines(fig, [a_true, b_true], color="C1")
corner.overplot_points(fig, np.array([a_true, b_true])[None], marker="s", color="C1")

# Example 1.b
Now what happens if we change the prior to the uniform priors. Recall that this was problematic for SBI with VI due to the limited flexibility of our surrogate

In [ ]:
def simulation(theta): 
    a, b = theta
    noise = yerr * np.random.normal(size=len(x))
    return a + b * x + noise

def sample_prior(): 
    # sample a and b from uniform prior
    _a = -0.5 + np.random.uniform() 
    _b = 10*np.random.uniform()
    return [_a, _b]

Nsample = 500000

samples = np.zeros((Nsample, 6))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.concatenate([theta_p, data_p])
samples = torch.tensor(samples, dtype=torch.float32)

In [ ]:
maf = NPE(density_estimator='maf')

# append (theta, D) from the simulations
_ = maf.append_simulations(samples[:,:2], samples[:,2:])
# train NPE. 
npe_maf = maf.train()

In [ ]:
posterior_maf = maf.build_posterior()
q_samples = posterior_maf.sample((10000,), x=torch.tensor(y, dtype=torch.float32))

In [ ]:
fig = plt.figure(figsize=(4,4))
fig = corner.corner(q_samples.numpy(), labels=['a', 'b'], fig=fig)
corner.overplot_lines(fig, [a_true, b_true], color="C1")
corner.overplot_points(fig, np.array([a_true, b_true])[None], marker="s", color="C1")

# In-class exercise: the stellar initial mass function
This time we'll infer two parameters of the IMF from a single star cluster:

- the slope $\alpha$ of $p(m) \propto m^{-\alpha}$;
- the **upper mass cutoff** $\log_{10} m_{\rm max}$: no star above it forms

Each cluster has $N = 500$ stars with masses between $1\,M_\odot$ and $m_{\rm max}$. The data are two summaries,

$$\mathcal{D} = \left(\overline{\log m},\ \log m_{\rm most\ massive}\right).$$


The priors are Gaussian, as if from earlier studies of other clusters:

$$\alpha \sim \mathcal{N}(2.35, 0.3^2), \qquad \log_{10} m_{\rm max} \sim \mathcal{N}(1.9, 0.2^2)\quad (m_{\rm max} \approx 50\text{–}125\,M_\odot \text{ at } 1\sigma).$$

*Note: The posterior will be exactly zero for $m_{\rm max}$ below the most massive observed star, because no star can be heavier than the cutoff.*

We'll follow the workflow on the slides:

1. sample $\{(\theta', \mathcal{D}')\} \sim p(\theta, \mathcal{D})$ with the simulator;
2. train **neural posterior estimation** (NPE) and **neural likelihood estimation** (NLE) with the `sbi` package, as in
   the demo above;
3. sample each posterior and compare it with the exact one.

In [ ]:
from torch.distributions import MultivariateNormal
from sbi.inference import NPE, NLE

torch.manual_seed(23)
rng = np.random.default_rng(23)

N_stars, m_lo = 500, 1.0

# prior on  alpha, log m_max
prior_mean, prior_std = np.array([2.35, 1.9]), np.array([0.3, 0.2])
prior = MultivariateNormal(torch.tensor(prior_mean, dtype=torch.float32),
                           torch.diag(torch.tensor(prior_std, dtype=torch.float32)**2))  

def sample_masses(alpha, m_max, n=N_stars):
    ''' inverse-CDF sampling of p(m) ~ m^-alpha on [m_lo, m_max] (Lecture 6) 
    '''
    u = rng.random(n)
    a1 = 1.0 - alpha
    return (m_lo**a1 + u * (m_max**a1 - m_lo**a1))**(1.0 / a1)

def summarize(m):
    return np.array([np.mean(np.log(m)), np.log(m.max())])

def simulator(theta):
    ''' theta: [n, 2] array of (alpha, log m_max) -> D: [n, 2] tensor 
    '''
    theta = np.atleast_2d(np.asarray(theta))
    return torch.tensor(np.array([summarize(sample_masses(a, 10**lM)) for a, lM in theta]), dtype=torch.float32)

This is again a toy case where we write down the exact posterior. Use the functions below for comparisons.

In [ ]:
# exact posterior on a grid in (alpha, log10 m_max)
a_grid = np.linspace(1.2, 3.5, 461)
lM_grid = np.linspace(1.0, 2.8, 451)
A, LM = np.meshgrid(a_grid, lM_grid, indexing='ij')

def exact_posterior(m):
    a1, M = 1.0 - A, 10**LM
    logZ = np.log((M**a1 - m_lo**a1) / a1)
    logp = -A * np.sum(np.log(m)) - len(m) * logZ
    logp += -0.5 * ((A - prior_mean[0]) / prior_std[0])**2 - 0.5 * ((LM - prior_mean[1]) / prior_std[1])**2
    logp[M < m.max()] = -np.inf          # no star above the cutoff
    p = np.exp(logp - logp.max())
    return p / p.sum()

Our mock observations. Three clusters with different $\alpha$ and $m_max$.

In [ ]:
theta_true = [(2.35, 60.0), (2.0, 120.0), (2.8, 40.0)]
clusters = []
for a, Mx in theta_true:
    m = sample_masses(a, Mx)
    clusters.append(dict(theta=(a, Mx), m=m, x=torch.tensor(summarize(m), dtype=torch.float32),
                         p_exact=exact_posterior(m)))

## 1. Simulations
Sample $(\theta', \mathcal{D}') \sim p(\theta, \mathcal{D})$, by drawing $\theta'$ from the
prior and running the simulator. Use the same 10,000 simulations for both NPE and NLE. Plot $p(\theta, \mathcal{D})$ using `corner`. 

## 2. Neural posterior estimation

Fit $q_\phi(\theta \mid \mathcal{D})$ directly by minimizing $-\sum \log q_\phi(\theta' \mid \mathcal{D}')$, which
minimizes $D_{\rm KL}(p(\theta \mid \mathcal{D}) \,\|\, q_\phi(\theta \mid \mathcal{D}))$ averaged over $p(\mathcal{D})$.

Estimate and plot the posterior for the three observed clusters: $q_\phi(\theta \mid \mathcal{D}_{\rm obs})$. 

*hint: once trained, the NPE is amortized. This means that the posterior for any new cluster is one pass through the network, with no new training or simulations.* 

## 3. Neural likelihood estimation

Fit $q_\phi(\mathcal{D} \mid \theta)$ instead.
To get the posterior, combine it with the prior and run **MCMC** on $q_\phi(\mathcal{D}_{\rm obs} \mid \theta)\,p(\theta)$.

*Hint 1: The likelihood estimator is amortized, BUT the MCMC must be rerun for every cluster.*

*Hint 2: use `emcee`. Don't forget to run all the convergence checks for MCMC*

## 4. Comparing NPE and NLE with the exact posterior
Compare and plot posteriors from NPE and NLE with the exact posterior for all three clusters. Include the true parameters in the plot as well.  

## (Optional) things to try
* try varying the architecture of the NDE
* repeat NPE and NLE with fewer (1,000) simulations instead of 10,000.